In [ ]:
%pip install -q unsloth "trl>=0.22.2,<0.25"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from sklearn.model_selection import train_test_split
from pathlib import Path
import json
import math
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1125 10:15:13.488000 500 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1125 10:15:13.538000 500 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
MAX_SEQ_LENGTH = 2048
DATA_PATH = "/content/drive/MyDrive/dataftune_tasks_only_refined_3430.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

tokenizer.padding_side = "right"

==((====))==  Unsloth 2026.11.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
ALLOWED_TOOLS = {"base", "retrieval", "database"}

def nonempty(value):
    return isinstance(value, str) and bool(value.strip())

def check_example(messages):
    if [m["role"] for m in messages] != ["user", "assistant"]:
        raise ValueError("Sai thứ tự user, assistant")
    user, answer = [json.loads(m["content"]) for m in messages]
    if set(user) != {"history", "question"} or not nonempty(user["question"]):
        raise ValueError("Sai định dạng đầu vào")
    if not isinstance(user["history"], list) or not all(
        turn["role"] in {"user", "assistant"} and nonempty(turn["content"])
        for turn in user["history"]
    ):
        raise ValueError("history không hợp lệ")
    if set(answer) != {"tasks"} or not isinstance(answer["tasks"], list) or not answer["tasks"]:
        raise ValueError("Đầu ra cần đúng một khóa tasks chứa ít nhất một tác vụ")
    tasks = answer["tasks"]
    for task in tasks:
        if (set(task) != {"tool", "question"} or task["tool"] not in ALLOWED_TOOLS
                or not nonempty(task["question"])):
            raise ValueError("Tác vụ không hợp lệ")
    return tasks[0]["tool"] if len(tasks) == 1 else f"multi_{len(tasks)}"

groups = []
for i, row in enumerate(dataset, start=1):
    try:
        groups.append(check_example(row["messages"]))
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(f"Mẫu {i}: {error}") from error

train_ids, val_ids = train_test_split(
    range(len(dataset)), test_size=0.10, stratify=groups, random_state=42
)
print(f"Train: {len(train_ids)} | Validation: {len(val_ids)}")

Generating train split: 0 examples [00:00, ? examples/s]

Train: 3087 | Validation: 343


In [ ]:
def render_batch(batch):
    texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]
    lengths = tokenizer(texts, add_special_tokens=False, return_length=True)["length"]
    return {"text": texts, "length": lengths}

dataset = dataset.map(render_batch, batched=True, remove_columns=dataset.column_names)
longest = max(dataset["length"])
if longest > MAX_SEQ_LENGTH:
    raise ValueError(f"Mẫu dài nhất có {longest} token. Tăng MAX_SEQ_LENGTH và nạp lại model.")
train_length = 8 * math.ceil(longest / 8)
dataset = dataset.remove_columns("length")
print(f"Mẫu dài nhất: {longest} token | Giới hạn khi train: {train_length}")

Map:   0%|          | 0/3430 [00:00<?, ? examples/s]

Mẫu dài nhất: 435 token | Giới hạn khi train: 440


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.11.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [ ]:
use_bf16 = is_bfloat16_supported()

# Cấu hình cho Colab một GPU. Giữ batch hiệu dụng 16 khi đổi GPU.
vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
micro_batch = 1 if vram_gib < 20 else 2
accum = 16 // micro_batch
epochs = 2
steps = math.ceil(len(train_ids) / (micro_batch * accum)) * epochs

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset.select(train_ids),
    eval_dataset=dataset.select(val_ids),
    args=SFTConfig(
        dataset_text_field="text",
        max_length=train_length,
        packing=False,
        per_device_train_batch_size=micro_batch,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=accum,
        num_train_epochs=epochs,
        warmup_steps=max(1, round(0.05 * steps)),
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=20,
        optim="adamw_8bit",
        output_dir="/content/qwen25-7b-router-checkpoints",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
    ),
)

# Chỉ tính loss trên câu trả lời JSON của assistant.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Kiểm tra nhãn thực tế trước khi cập nhật trọng số.
sample = trainer.train_dataset[0]
target = tokenizer.decode([t for t in sample["labels"] if t != -100], skip_special_tokens=True)
assert json.loads(target) == json.loads(tokenizer.decode(
    sample["input_ids"], skip_special_tokens=False
).rsplit("<|im_start|>assistant\n", 1)[1].split("<|im_end|>", 1)[0]), "Kiểm tra lại mask của assistant"

trainer_stats = trainer.train()
print("Checkpoint tốt nhất:", trainer.state.best_model_checkpoint)
adapter_dir = "/content/drive/MyDrive/qwen25-7b-router-lora"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3087 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/343 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/3087 [00:00<?, ? examples/s]

Map:   0%|          | 0/343 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,087 | Num Epochs = 2 | Total steps = 386
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


... (3 outputs left out)

Unsloth: Restored added_tokens_decoder metadata in /content/qwen25-7b-router-checkpoints/checkpoint-193/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/qwen25-7b-router-checkpoints/checkpoint-386/tokenizer_config.json.


Checkpoint tốt nhất: /content/qwen25-7b-router-checkpoints/checkpoint-386


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen25-7b-router-lora/tokenizer_config.json.


('/content/drive/MyDrive/qwen25-7b-router-lora/tokenizer_config.json',
 '/content/drive/MyDrive/qwen25-7b-router-lora/chat_template.jinja',
 '/content/drive/MyDrive/qwen25-7b-router-lora/tokenizer.json')

In [ ]:
# Tùy chọn: chạy các cell còn lại nếu cần GGUF Q4_K_M.
export_dir = Path("/content/qwen25-7b-router")
model.save_pretrained_gguf(str(export_dir), tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/qwen25-7b-router/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (15 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:56<00:00, 89.07s/it]


Unsloth: Merge process complete. Saved to `/content/qwen25-7b-router`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12676-mix-c32dcc8 (app-b12676-mix-c32dcc8-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/qwen25-7b-router_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions

{'save_directory': '/content/qwen25-7b-router',
 'gguf_directory': '/content/qwen25-7b-router_gguf',
 'gguf_files': ['/content/qwen25-7b-router_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/qwen25-7b-router_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# Unsloth có thể xuất GGUF vào thư mục model hoặc thư mục có hậu tố _gguf.
ggufs = [p for folder in (export_dir, Path(str(export_dir) + "_gguf"))
         for p in folder.rglob("*.gguf") if "q4_k_m" in p.name.lower()]
if not ggufs:
    raise FileNotFoundError("Không tìm thấy tệp GGUF Q4_K_M sau khi xuất.")

In [ ]:
from zipfile import ZipFile, ZIP_STORED

archive = Path("/content/drive/MyDrive/qwen25-7b-router-q4_k_m.zip")
with ZipFile(archive, "w", compression=ZIP_STORED) as bundle:
    for path in ggufs:
        bundle.write(path, arcname=path.name)
print(archive)

/content/drive/MyDrive/qwen25-7b-router-q4_k_m.zip


In [ ]:
# Tùy chọn: tải bản GGUF đã lưu trên Drive về máy.
from google.colab import files
files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>